In [2]:
import pandas as pd

df = pd.read_csv('stress_dataset_final_merged.csv')

print(df.head())

                                                text   label
0  Nevermind, Every other weekend... I googled it...  medium
1  [NAME] was arrested for lying and obstruction,...  medium
2        Well damn I’m going to the Buckingham today  medium
3  These are both beauties. Those big dark eyes a...     low
4                                 ! remind me 2 days  medium


In [3]:
print("="*50)
print("Dataset Info")
print("="*50)

print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
print(f"Columns: {list(df.columns)}\n\n")
df.info()

Dataset Info
Rows: 21436, Columns: 2
Columns: ['text', 'label']


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21436 entries, 0 to 21435
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    21436 non-null  object
 1   label   21436 non-null  object
dtypes: object(2)
memory usage: 335.1+ KB


In [4]:
# Check Null & Missing Values
print("="*50)
print("Missing Values")
print("="*50)

print(df.isnull().sum())
print(f"\nTotal missing cells: {df.isnull().sum().sum()}")

print("\nEMPTY STRINGS:")
for col in df.select_dtypes(include='object').columns:
    empty_count = (df[col].str.strip() == '').sum()
    print(f" {col}: {empty_count} empty strings")

Missing Values
text     0
label    0
dtype: int64

Total missing cells: 0

EMPTY STRINGS:
 text: 0 empty strings
 label: 0 empty strings


In [5]:
# Check Duplicates
print("="*50)
print("DUPLICATES")
print("="*50)

print(f"Duplicate rows: {df.duplicated().sum()}")
print(f"Duplicate text only: {df['text'].duplicated().sum()}")

DUPLICATES
Duplicate rows: 0
Duplicate text only: 1


In [6]:
# Check Label Distribution & Validity
print("="*50)
print("Label Distribution")
print("="*50)

print(df['label'].value_counts())

print(f"\nLabels: {df['label'].unique()}")
expected_labels = {'high', 'medium', 'low'}
unexpected = set(df['label'].unique()) - expected_labels

print(f"\nUnexpected Labels: {unexpected if unexpected else 'None'}")

print(f"\nLabels with leading/trailing spaces: {(df['label'] != df['label'].str.strip()).sum()}")

Label Distribution
label
high      7883
medium    6792
low       6761
Name: count, dtype: int64

Labels: ['medium' 'low' 'high']

Unexpected Labels: None

Labels with leading/trailing spaces: 0


In [7]:
# Check Text Length
print("="*50)
print("Text Length")
print("="*50)

df["text_len"] = df["text"].astype(str).apply(len)
short_texts = df[df["text_len"] < 10]
print(f"\nTexts shorter than 10 chars: {len(short_texts)}")

if len(short_texts) > 0:
    print(short_texts[["text", "label"]].head())

long_texts = df[df["text_len"] > 1000]
print(f"\nTexts longer than 1000 chars: {len(long_texts)}")

if len(long_texts) > 0:
    print(long_texts[["text", "label"]].head())

Text Length

Texts shorter than 10 chars: 0

Texts longer than 1000 chars: 616
                                                   text label
3624  Weird title I know but I really couldn’t find ...  high
3652  so i finally got tired of being anxious and de...  high
3687  He forced me into sex again... And if I refuse...  high
3694  I just couldn't cope, the abuse I already reme...  high
3725  Hi everyone, this is my story, what I've been ...  high


In [8]:
# Check Special Characters & Noise
print("="*50)
print("Check Noise")
print("="*50)

print(f"Texts with URLs:      {df['text'].str.contains(r'http\S+|www\S+', na=False).sum()}")
print(f"Texts with @mentions: {df['text'].str.contains(r'@\w+', na=False).sum()}")
print(f"Texts with #hashtags: {df['text'].str.contains(r'#\w+', na=False).sum()}")
print(f"Texts with numbers:   {df['text'].str.contains(r'\d', na=False).sum()}")
print(f"Texts with emojis:    {df['text'].str.contains(r'[^\x00-\x7F]', na=False).sum()}")

Check Noise
Texts with URLs:      8
Texts with @mentions: 375
Texts with #hashtags: 64
Texts with numbers:   5526
Texts with emojis:    3250


In [9]:
# Check Balance
print("="*50)
print("Class Balance")
print("="*50)
counts = df['label'].value_counts()
total = len(df)
for label, count in counts.items():
    print(f"  {label}: {count} ({count/total*100:.1f}%)")

imbalance_ratio = counts.max() / counts.min()
print(f"\nImbalance ratio (max/min): {imbalance_ratio:.2f}")
if imbalance_ratio > 1.5:
    print("Consider handling class imbalance")
else:
    print("Classes are balanced")

Class Balance
  high: 7883 (36.8%)
  medium: 6792 (31.7%)
  low: 6761 (31.5%)

Imbalance ratio (max/min): 1.17
Classes are balanced


In [10]:
# Check Encoding Issues
print("="*50)
print("Encoding Issues")
print("="*50)

weird = df['text'].str.contains(r'[^\x00-\x7F]', na=False)
print(f"\nRows with non-ASCII characters: {weird.sum()}")
if weird.sum() > 0:
    print("\nSample non-ASCII rows:")
    print(df[weird]['text'].head().values)

Encoding Issues

Rows with non-ASCII characters: 3250

Sample non-ASCII rows:
['Well damn I’m going to the Buckingham today'
 'The only flight delay I’ve had at slc is because there was too much rain at lax'
 'This was probably one of the best videos I’ve ever seen on Reddit. What an amazing story. Job well done. RIP Chief.'
 'I guess it’s no surprise that [NAME] really was [NAME] Destenay.'
 'dude, your mom is a fuck-nugget, i’m so sorry you’re dealing with this ']


SOLVE THE ISSUES

In [11]:
import re

def clean_text(text):
    text = str(text).lower()                          
    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)        
    # Remove @mentions
    text = re.sub(r'@\w+', '', text)                  
    # Remove #hashtags
    text = re.sub(r'#\w+', '', text)                  
    # Remove numbers
    text = re.sub(r'\d+', '', text)                   
    # Remove punctuation
    text = re.sub(r'[^\w\s]', '', text)               
    # Remove emojis
    text = re.sub(r'[^\x00-\x7F]+', '', text)  
    # Normalize spaces 
    text = re.sub(r'\s+', ' ', text).strip()  
    return text

df['cleaned_text'] = df['text'].apply(clean_text)

In [12]:
# Remove Duplicate Text
df = df.drop_duplicates(subset=['text']).reset_index(drop=True)

print(f"Rows after removing duplicates: {len(df)}")

Rows after removing duplicates: 21435


In [13]:
# Remove Short or Empty Texts
df = df[df['cleaned_text'].str.len() >= 10].reset_index(drop=True)
print("Rows after filtering:", len(df))

Rows after filtering: 21431


In [14]:
# Encode Labels
LABEL_MAPPING = {
    "low": 0,
    "medium": 1,
    "high": 2
}

df["label_encoded"] = df["label"].map(LABEL_MAPPING)

if df["label_encoded"].isnull().sum() > 0:
    print("Some labels were not mapped correctly")

print(df.head())
print(df["label_encoded"].value_counts())

                                                text   label  text_len  \
0  Nevermind, Every other weekend... I googled it...  medium        84   
1  [NAME] was arrested for lying and obstruction,...  medium        84   
2        Well damn I’m going to the Buckingham today  medium        43   
3  These are both beauties. Those big dark eyes a...     low        60   
4                                 ! remind me 2 days  medium        18   

                                        cleaned_text  label_encoded  
0  nevermind every other weekend i googled it and...              1  
1  name was arrested for lying and obstruction to...              1  
2         well damn im going to the buckingham today              1  
3  these are both beauties those big dark eyes an...              0  
4                                     remind me days              1  
label_encoded
2    7883
1    6791
0    6757
Name: count, dtype: int64


In [15]:
# Lemmatization
from nltk.stem import WordNetLemmatizer
import nltk
nltk.download('wordnet')
nltk.download('omw-1.4')

wnl = WordNetLemmatizer()

def lemmatize_text(text):
    return ' '.join([wnl.lemmatize(word, pos="v") for word in text.split()])

df['cleaned_text'] = df['cleaned_text'].apply(lemmatize_text)
df['cleaned_text'].head()

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\yomna\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\yomna\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


0    nevermind every other weekend i google it and ...
1    name be arrest for lie and obstruction to fbi ...
2              well damn im go to the buckingham today
3    these be both beauties those big dark eye and ...
4                                       remind me days
Name: cleaned_text, dtype: object

In [16]:
# Tokenization
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

from nltk.tokenize import word_tokenize

def tokenize_text(text):
    return ' '.join(word_tokenize(text))

df['cleaned_text'] = df['cleaned_text'].apply(tokenize_text)
df['cleaned_text'].head()

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\yomna\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\yomna\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


0    nevermind every other weekend i google it and ...
1    name be arrest for lie and obstruction to fbi ...
2              well damn im go to the buckingham today
3    these be both beauties those big dark eye and ...
4                                       remind me days
Name: cleaned_text, dtype: object

In [17]:
# Remove Stopwords
stop_words = {
    'i', 'me', 'my', 'we', 'you', 'he', 'she', 'it', 'they', 'what',
    'this', 'that', 'am', 'is', 'are', 'was', 'were', 'be', 'been',
    'have', 'has', 'had', 'do', 'does', 'did', 'will', 'would', 'shall',
    'should', 'can', 'could', 'a', 'an', 'the', 'and', 'but', 'if', 'or',
    'of', 'at', 'by', 'for', 'with', 'to', 'from', 'in', 'out', 'on',
    'so', 'than', 'too', 'very', 'just', 'not', 'no', 'only', 'also'
}

def remove_stopwords(text):
    return ' '.join([word for word in text.split() if word not in stop_words])

df['cleaned_text'] = df['cleaned_text'].apply(remove_stopwords)

In [18]:
df['word_count']      = df['cleaned_text'].str.split().str.len()
df['text_length']     = df['cleaned_text'].str.len()
df['avg_word_length'] = df['cleaned_text'].apply(lambda x: sum(len(w) for w in x.split()) / len(x.split()) if x.split() else 0)
df['punct_count']     = df['text'].apply(lambda x: sum(1 for c in x if c in '!?.,;:'))
df['uppercase_ratio'] = df['text'].apply(lambda x: sum(1 for c in x if c.isupper()) / len(x) if len(x) > 0 else 0)
df

,text,label,text_len,cleaned_text,label_encoded,word_count,text_length,avg_word_length,punct_count,uppercase_ratio
0,"Nevermind, Every other weekend... I googled it...",medium,84,nevermind every other weekend google didnt thi...,1,10,63,5.400000,4,0.035714
1,"[NAME] was arrested for lying and obstruction,...",medium,84,name arrest lie obstruction fbi collusion calm...,1,9,56,5.333333,2,0.083333
2,Well damn I’m going to the Buckingham today,medium,43,well damn im go buckingham today,1,6,32,4.500000,0,0.069767
3,These are both beauties. Those big dark eyes a...,low,60,these both beauties those big dark eye smile,0,8,44,4.625000,2,0.033333
4,! remind me 2 days,medium,18,remind days,1,2,11,5.000000,1,0.000000
...,...,...,...,...,...,...,...,...,...,...
21426,I have bipolar 1 and panic disorder and when m...,high,281,bipolar panic disorder when panic disorder sta...,2,29,198,5.862069,6,0.010676
21427,When Im stressed which is most of the time my ...,medium,67,when im stress which most time anger get worse,1,9,46,4.222222,0,0.029851
21428,Not exactly helpful for single parents overwhe...,high,174,exactly helpful single parent overwhelm even a...,2,20,120,5.050000,3,0.017241
21429,Without school you'll get even more stress at ...,medium,54,without school youll get even more stress end,1,8,45,4.750000,1,0.018519


In [19]:
df = df.drop(columns=['text', 'label', 'text_len'])

print(df.head())

                                        cleaned_text  label_encoded  \
0  nevermind every other weekend google didnt thi...              1   
1  name arrest lie obstruction fbi collusion calm...              1   
2                   well damn im go buckingham today              1   
3       these both beauties those big dark eye smile              0   
4                                        remind days              1   

   word_count  text_length  avg_word_length  punct_count  uppercase_ratio  
0          10           63         5.400000            4         0.035714  
1           9           56         5.333333            2         0.083333  
2           6           32         4.500000            0         0.069767  
3           8           44         4.625000            2         0.033333  
4           2           11         5.000000            1         0.000000  


In [20]:
df.to_csv('df_preprocessed.csv', index=False)

CHECK THE DATASET AFTER PREPROCESSING

In [21]:
import pandas as pd

df = pd.read_csv('df_preprocessed.csv')

print(df.head())

                                        cleaned_text  label_encoded  \
0  nevermind every other weekend google didnt thi...              1   
1  name arrest lie obstruction fbi collusion calm...              1   
2                   well damn im go buckingham today              1   
3       these both beauties those big dark eye smile              0   
4                                        remind days              1   

   word_count  text_length  avg_word_length  punct_count  uppercase_ratio  
0          10           63         5.400000            4         0.035714  
1           9           56         5.333333            2         0.083333  
2           6           32         4.500000            0         0.069767  
3           8           44         4.625000            2         0.033333  
4           2           11         5.000000            1         0.000000  


In [22]:
print("="*50)
print("Dataset Info")
print("="*50)

print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
print(f"Columns: {list(df.columns)}\n\n")
df.info()

Dataset Info
Rows: 21431, Columns: 7
Columns: ['cleaned_text', 'label_encoded', 'word_count', 'text_length', 'avg_word_length', 'punct_count', 'uppercase_ratio']


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21431 entries, 0 to 21430
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   cleaned_text     21429 non-null  object 
 1   label_encoded    21431 non-null  int64  
 2   word_count       21431 non-null  int64  
 3   text_length      21431 non-null  int64  
 4   avg_word_length  21431 non-null  float64
 5   punct_count      21431 non-null  int64  
 6   uppercase_ratio  21431 non-null  float64
dtypes: float64(2), int64(4), object(1)
memory usage: 1.1+ MB


In [23]:
df = df.dropna(subset=['cleaned_text']).reset_index(drop=True)

print(f"Rows after dropping nulls: {len(df)}")

Rows after dropping nulls: 21429


In [24]:
# Check Null & Missing Values
print("="*50)
print("Missing Values")
print("="*50)

print(df.isnull().sum())
print(f"\nTotal missing cells: {df.isnull().sum().sum()}")

print("\nEMPTY STRINGS:")
for col in df.select_dtypes(include='object').columns:
    empty_count = (df[col].str.strip() == '').sum()
    print(f" {col}: {empty_count} empty strings")

Missing Values
cleaned_text       0
label_encoded      0
word_count         0
text_length        0
avg_word_length    0
punct_count        0
uppercase_ratio    0
dtype: int64

Total missing cells: 0

EMPTY STRINGS:
 cleaned_text: 0 empty strings


In [25]:
# Remove Duplicate Text
df = df.drop_duplicates(subset=['cleaned_text']).reset_index(drop=True)

print(f"Rows after removing duplicates: {len(df)}")

Rows after removing duplicates: 21217


In [26]:
# Check Duplicates
print("="*50)
print("DUPLICATES")
print("="*50)

print(f"Duplicate rows: {df.duplicated().sum()}")
print(f"Duplicate text only: {df['cleaned_text'].duplicated().sum()}")

DUPLICATES
Duplicate rows: 0
Duplicate text only: 0


In [27]:
# Check Special Characters & Noise
print("="*50)
print("Check Noise")
print("="*50)

print(f"Texts with URLs:      {df['cleaned_text'].str.contains(r'http\S+|www\S+', na=False).sum()}")
print(f"Texts with @mentions: {df['cleaned_text'].str.contains(r'@\w+', na=False).sum()}")
print(f"Texts with #hashtags: {df['cleaned_text'].str.contains(r'#\w+', na=False).sum()}")
print(f"Texts with numbers:   {df['cleaned_text'].str.contains(r'\d', na=False).sum()}")
print(f"Texts with emojis:    {df['cleaned_text'].str.contains(r'[^\x00-\x7F]', na=False).sum()}")

Check Noise
Texts with URLs:      0
Texts with @mentions: 0
Texts with #hashtags: 0
Texts with numbers:   0
Texts with emojis:    0


In [28]:
df = df[df['cleaned_text'].str.len() >= 10].reset_index(drop=True)
print(f"Rows after removing short texts: {len(df)}")

df = df[df['cleaned_text'].str.len() <= 1000].reset_index(drop=True)
print(f"Rows after capping long texts: {len(df)}")

Rows after removing short texts: 21180
Rows after capping long texts: 20965


In [32]:
print(f"Shape: {df.shape}")
print(f"Nulls: {df.isnull().sum().sum()}")
print(f"Duplicates: {df.duplicated().sum()}")
print(f"Duplicate cleaned_text: {df['cleaned_text'].duplicated().sum()}")
print(f"Min text length: {df['cleaned_text'].str.len().min()}")
print(f"Max text length: {df['cleaned_text'].str.len().max()}")
print(f"Columns: {list(df.columns)}")

Shape: (20965, 7)
Nulls: 0
Duplicates: 0
Duplicate cleaned_text: 0
Min text length: 10
Max text length: 998
Columns: ['cleaned_text', 'label_encoded', 'word_count', 'text_length', 'avg_word_length', 'punct_count', 'uppercase_ratio']


In [30]:
df.to_csv('df_preprocessed.csv', index=False)

SPLIT THE DATASET

In [31]:
# Split Dataset
from sklearn.model_selection import train_test_split

X = df[['cleaned_text']]
y = df['label_encoded']

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=100)

print(f"Train: {len(X_train)} | Test: {len(X_test)}")

X_train.to_csv('X_train.csv', index=False)
X_test.to_csv('X_test.csv', index=False)
y_train.to_csv('y_train.csv', index=False)
y_test.to_csv('y_test.csv', index=False)

Train: 15723 | Test: 5242
